# ViT CLAHE 10에포크 복구

기존 실행이 종료된 후 마지막 완료 에포크의 체크포인트에서 두 조건을 총 10에포크까지 이어서 학습합니다. Drive의 기존 체크포인트와 원본 ZIP을 재사용합니다. 초기 3에포크 실행 셀을 다시 실행하지 않습니다.

In [ ]:
import os
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
import torch
assert torch.cuda.is_available(), '런타임 유형을 GPU로 바꿔주세요.'
print(torch.__version__, torch.cuda.get_device_name(0))
%pip -q install transformers==4.57.6 opencv-python-headless pandas scikit-learn matplotlib


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import zipfile
import os
import shutil
workspace = Path('/content/cxr_clahe')
archive = Path('/content/drive/MyDrive/CXR_CLAHE/vit_clahe_colab.zip')
assert archive.is_file(), f'{archive}에 ZIP을 업로드하세요.'
workspace.mkdir(exist_ok=True)
marker = workspace / '.extracted'
if not marker.exists():
    assert shutil.disk_usage('/content').free > archive.stat().st_size + 2_000_000_000
    with zipfile.ZipFile(archive) as package:
        package.extractall(workspace)
    marker.touch()
os.chdir(workspace)
output_root = Path('/content/drive/MyDrive/CXR_CLAHE/results')
output_root.mkdir(parents=True, exist_ok=True)


In [ ]:
import json
sources = json.loads('{"dataset.py": "\\"\\"\\"Load images from an explicit, QC-approved manifest.\\"\\"\\"\\nfrom pathlib import Path\\nimport pandas as pd\\nfrom PIL import Image\\nimport torch\\nfrom torch.utils.data import Dataset\\n\\n\\nclass ChestXrayDataset(Dataset):\\n    def __init__(self, manifest_path, image_dir, transform=None):\\n        self.table = pd.read_csv(manifest_path, dtype={\'Patient ID\':str})\\n        required = {\'Image Index\', \'Patient ID\', \'pneumonia\'}\\n        if not required.issubset(self.table.columns):\\n            raise ValueError(\'Manifest lacks required columns.\')\\n        if self.table[\'Image Index\'].duplicated().any():\\n            raise ValueError(\'Manifest contains duplicate filenames.\')\\n        if not self.table[\'pneumonia\'].isin([0,1]).all():\\n            raise ValueError(\'Targets must be 0 or 1.\')\\n        self.image_dir = Path(image_dir)\\n        self.transform = transform\\n\\n    def __len__(self):\\n        return len(self.table)\\n\\n    def __getitem__(self, index):\\n        row = self.table.iloc[index]\\n        with Image.open(self.image_dir / row[\'Image Index\']) as source:\\n            image = source.copy()\\n        if self.transform is not None:\\n            image = self.transform(image)\\n        return image, torch.tensor(float(row[\'pneumonia\']), dtype=torch.float32)\\n", "evaluate.py": "\\"\\"\\"Evaluate a completed ViT checkpoint on a fixed cohort.\\"\\"\\"\\nimport argparse\\nimport hashlib\\nimport json\\nimport time\\nfrom pathlib import Path\\n\\nimport pandas as pd\\nimport torch\\nfrom sklearn.metrics import average_precision_score, roc_auc_score\\nfrom torch.utils.data import DataLoader\\nfrom transformers import ViTConfig, ViTForImageClassification, ViTImageProcessor\\n\\nfrom src.dataset import ChestXrayDataset\\nfrom src.preprocessing import ApplyCLAHE\\nfrom src.train import ROOT\\nfrom src.vit_model import ViTBinaryClassifier\\n\\n\\ndef main():\\n    parser = argparse.ArgumentParser(description=__doc__)\\n    parser.add_argument(\'run_dir\', type=Path)\\n    parser.add_argument(\'--split\', choices=(\'val\', \'test\'), required=True)\\n    parser.add_argument(\'--device\', choices=(\'cpu\', \'cuda\'), default=\'cpu\')\\n    args = parser.parse_args()\\n    if args.device == \'cuda\' and not torch.cuda.is_available():\\n        parser.error(\'CUDA GPU unavailable\')\\n    device = torch.device(args.device)\\n    run = args.run_dir\\n    if not (run / \'completed.json\').is_file():\\n        parser.error(\'Choose a completed training run.\')\\n    output = run / f\'{args.split}_evaluation\'\\n    if output.exists():\\n        parser.error(f\'Evaluation already exists: {output}\')\\n\\n    checkpoint = torch.load(run / \'best.pt\', weights_only=True, map_location=\'cpu\')\\n    config = checkpoint[\'config\']\\n    if config[\'smoke\'] or config[\'model\'] != \'vit\':\\n        parser.error(\'Choose a full ViT training checkpoint.\')\\n    torch.set_num_threads(config[\'threads\'])\\n\\n    manifests = ROOT / \'data/processed/reduced_train_full_eval_qc\'\\n    cohort_path = manifests / f\'{args.split}.csv\'\\n    for split in (\'train\', \'val\'):\\n        actual = hashlib.sha256((manifests / f\'{split}.csv\').read_bytes()).hexdigest()\\n        if actual != config[\'manifest_sha256\'][split]:\\n            raise ValueError(f\'{split} manifest differs from the training run.\')\\n\\n    processor = ViTImageProcessor.from_pretrained(run, local_files_only=True)\\n    model_config = ViTConfig.from_pretrained(\\n        config[\'pretrained_model\'], revision=config[\'revision\'],\\n        cache_dir=ROOT / \'outputs/models/hf_cache\', local_files_only=True,\\n        num_labels=1,\\n    )\\n    model = ViTBinaryClassifier(ViTForImageClassification(model_config), False)\\n    model.load_state_dict(checkpoint[\'model_state_dict\'])\\n    model.to(device)\\n    model.eval()\\n\\n    clahe = ApplyCLAHE()\\n    def transform(image):\\n        if config.get(\\"clahe\\", False):\\n            image = clahe(image)\\n        return processor(image.convert(\'RGB\'), return_tensors=\'pt\')[\'pixel_values\'][0]\\n\\n    cohort = ChestXrayDataset(cohort_path, ROOT / \'data/raw/images\', transform)\\n    for split in (\'train\', \'val\'):\\n        if split != args.split:\\n            patients = pd.read_csv(manifests / f\'{split}.csv\', dtype={\'Patient ID\': str})[\'Patient ID\']\\n            if set(patients) & set(cohort.table[\'Patient ID\']):\\n                raise ValueError(f\'{split} and {args.split} patients overlap.\')\\n\\n    loader = DataLoader(cohort, batch_size=config[\'batch_size\'], shuffle=False)\\n    scores = []\\n    started = time.perf_counter()\\n    print(f\'Evaluating {len(cohort)} {args.split} images, checkpoint epoch {checkpoint[\\"epoch\\"]}\', flush=True)\\n    with torch.inference_mode():\\n        for batch, (images, _) in enumerate(loader, 1):\\n            logits = model(images.to(device)).squeeze(1)\\n            if not torch.isfinite(logits).all():\\n                raise ValueError(f\'Non-finite scores at batch {batch}.\')\\n            scores.extend(logits.sigmoid().cpu().tolist())\\n            if batch % 100 == 0:\\n                print(f\'Batch {batch}/{len(loader)}\', flush=True)\\n\\n    rows = cohort.table[[\'Image Index\', \'Patient ID\', \'pneumonia\']].copy()\\n    rows[\'score\'] = scores\\n    targets = rows[\'pneumonia\']\\n    metrics = {\\n        \'split\': args.split,\\n        \'checkpoint_epoch\': checkpoint[\'epoch\'],\\n        \'manifest_sha256\': hashlib.sha256(cohort_path.read_bytes()).hexdigest(),\\n        \'images\': len(rows),\\n        \'patients\': int(rows[\'Patient ID\'].nunique()),\\n        \'positive_images\': int(targets.sum()),\\n        \'auroc\': roc_auc_score(targets, scores),\\n        \'average_precision\': average_precision_score(targets, scores),\\n        \'seconds\': time.perf_counter() - started,\\n    }\\n    output.mkdir()\\n    rows.to_csv(output / \'predictions.csv\', index=False)\\n    (output / \'metrics.json\').write_text(json.dumps(metrics, indent=2), encoding=\'utf-8\')\\n    print(json.dumps(metrics, indent=2), flush=True)\\n\\n\\nif __name__ == \'__main__\':\\n    main()\\n", "extend_training.py": "\\"\\"\\"Extend a ViT run from its last completed epoch into a separate output directory.\\"\\"\\"\\nimport argparse\\nimport hashlib\\nimport json\\nimport random\\nimport time\\nfrom pathlib import Path\\n\\nimport numpy as np\\nimport pandas as pd\\nimport torch\\nfrom torch import nn\\nfrom torch.utils.data import DataLoader, Subset\\n\\nfrom src.dataset import ChestXrayDataset\\nfrom src.preprocessing import ApplyCLAHE\\nfrom src.train import ROOT, run_epoch, save_checkpoint\\nfrom src.vit_model import load_vit\\n\\n\\ndef main():\\n    parser = argparse.ArgumentParser(description=__doc__)\\n    parser.add_argument(\'source\', type=Path)\\n    parser.add_argument(\'--epochs\', type=int, default=10)\\n    parser.add_argument(\'--output\', type=Path, required=True)\\n    args = parser.parse_args()\\n    source = args.source.resolve()\\n    output = args.output.resolve()\\n    if source == output:\\n        parser.error(\'Use a distinct output directory.\')\\n    # Retry an interrupted extension from its own last completed epoch.\\n    origin = output if (output / \'last.pt\').is_file() else source\\n    checkpoint = torch.load(origin / \'last.pt\', weights_only=True, map_location=\'cpu\')\\n    config = checkpoint[\'config\'].copy()\\n    if config.get(\'model\') != \'vit\':\\n        parser.error(\'Only ViT extensions are supported.\')\\n    if args.epochs <= checkpoint[\'epoch\']:\\n        parser.error(\'Target epochs must exceed the saved epoch.\')\\n    device = torch.device(config[\'device\'])\\n    if device.type == \'cuda\' and not torch.cuda.is_available():\\n        parser.error(\'Source run requires a CUDA GPU.\')\\n    random.seed(config[\'seed\'])\\n    np.random.seed(config[\'seed\'])\\n    torch.manual_seed(config[\'seed\'])\\n    torch.set_num_threads(config[\'threads\'])\\n    torch.use_deterministic_algorithms(True)\\n    if device.type == \'cuda\':\\n        torch.backends.cuda.matmul.allow_tf32 = False\\n        torch.backends.cudnn.allow_tf32 = False\\n\\n    manifest_dir = ROOT / \'data/processed/reduced_train_full_eval_qc\'\\n    for split, digest in config[\'manifest_sha256\'].items():\\n        if hashlib.sha256((manifest_dir / f\'{split}.csv\').read_bytes()).hexdigest() != digest:\\n            raise ValueError(\'Manifest changed.\')\\n    model, processor = load_vit(ROOT / \'outputs/models/hf_cache\', config[\'vit_last_block\'])\\n    clahe = ApplyCLAHE()\\n\\n    def transform(image):\\n        if config[\'clahe\']:\\n            image = clahe(image)\\n        return processor(image.convert(\'RGB\'), return_tensors=\'pt\')[\'pixel_values\'][0]\\n\\n    train = ChestXrayDataset(manifest_dir / \'train.csv\', ROOT / \'data/raw/images\', transform)\\n    if config[\'smoke\']:\\n        indices = torch.randperm(len(train), generator=torch.Generator().manual_seed(config[\'seed\']))[:32].tolist()\\n        train = Subset(train, indices)\\n        val = train\\n    else:\\n        val = ChestXrayDataset(manifest_dir / \'val.csv\', ROOT / \'data/raw/images\', transform)\\n        assert not set(train.table[\'Patient ID\']) & set(val.table[\'Patient ID\'])\\n    assert len(train) == config[\'train_images\'] and len(val) == config[\'val_images\']\\n    generator = torch.Generator()\\n    generator.set_state(checkpoint[\'loader_rng_state\'])\\n    train_loader = DataLoader(train, batch_size=config[\'batch_size\'], shuffle=True, generator=generator, num_workers=0)\\n    val_loader = DataLoader(val, batch_size=config[\'batch_size\'], shuffle=False, num_workers=0)\\n    model.load_state_dict(checkpoint[\'model_state_dict\'])\\n    model.to(device)\\n    optimizer = torch.optim.Adam((p for p in model.parameters() if p.requires_grad), lr=config[\'lr\'])\\n    optimizer.load_state_dict(checkpoint[\'optimizer_state_dict\'])\\n    torch.set_rng_state(checkpoint[\'torch_rng_state\'])\\n    if device.type == \'cuda\' and \'cuda_rng_states\' in checkpoint:\\n        torch.cuda.set_rng_state_all(checkpoint[\'cuda_rng_states\'])\\n    history = pd.read_csv(origin / \'history.csv\')\\n    history = history[history.epoch <= checkpoint[\'epoch\']].to_dict(\'records\')\\n    if [int(row[\'epoch\']) for row in history] != list(range(1, checkpoint[\'epoch\'] + 1)):\\n        raise ValueError(\'Incomplete source history.\')\\n    best_checkpoint = torch.load(origin / \'best.pt\', weights_only=True, map_location=\'cpu\')\\n    best = best_checkpoint[\'metrics\'][\'val_auroc\']\\n    start_epoch = checkpoint[\'epoch\'] + 1\\n    if output.exists() and origin != output:\\n        parser.error(\'Output already exists without a resumable checkpoint.\')\\n    output.mkdir(parents=True, exist_ok=True)\\n    config.update(epochs=args.epochs, run_name=output.name, resumed_from=str(source),\\n                  continuation_start_epoch=start_epoch)\\n    processor.save_pretrained(output)\\n    (output / \'config.json\').write_text(json.dumps(config, indent=2), encoding=\'utf-8\')\\n    if origin != output:\\n        save_checkpoint(output / \'best.pt\', best_checkpoint)\\n    del checkpoint, best_checkpoint\\n    criterion = nn.BCEWithLogitsLoss()\\n    for epoch in range(start_epoch, args.epochs + 1):\\n        started = time.perf_counter()\\n        print(f\'Epoch {epoch}/{args.epochs}, CLAHE={config[\\"clahe\\"]}\', flush=True)\\n        training = run_epoch(model, train_loader, criterion, optimizer, type(model).train_selected_layers)\\n        validation = run_epoch(model, val_loader, criterion)\\n        row = dict(epoch=epoch, seconds=time.perf_counter() - started,\\n                   **{f\'train_{key}\': value for key, value in training.items()},\\n                   **{f\'val_{key}\': value for key, value in validation.items()})\\n        history.append(row)\\n        pd.DataFrame(history).to_csv(output / \'history.csv\', index=False)\\n        checkpoint = dict(model_state_dict=model.state_dict(), optimizer_state_dict=optimizer.state_dict(),\\n                          epoch=epoch, config=config, metrics=row,\\n                          torch_rng_state=torch.get_rng_state(), loader_rng_state=generator.get_state())\\n        if device.type == \'cuda\':\\n            checkpoint[\'cuda_rng_states\'] = torch.cuda.get_rng_state_all()\\n        if row[\'val_auroc\'] is not None and row[\'val_auroc\'] > best:\\n            best = row[\'val_auroc\']\\n            save_checkpoint(output / \'best.pt\', checkpoint)\\n        save_checkpoint(output / \'last.pt\', checkpoint)\\n        print(json.dumps(row), flush=True)\\n    (output / \'completed.json\').write_text(json.dumps(dict(epochs=args.epochs, best_auroc=best)), encoding=\'utf-8\')\\n    print(f\'Completed {output}\', flush=True)\\n\\n\\nif __name__ == \'__main__\':\\n    main()\\n", "gradcam.py": "\\"\\"\\"Grad-CAM for the selected ViT, sampled from each test confusion group.\\"\\"\\"\\nimport hashlib\\nimport json\\n\\nimport matplotlib\\nmatplotlib.use(\'Agg\')\\nimport matplotlib.pyplot as plt\\nimport numpy as np\\nimport pandas as pd\\nimport torch\\nimport torch.nn.functional as F\\nfrom PIL import Image\\nfrom transformers import ViTConfig, ViTForImageClassification, ViTImageProcessor\\n\\nfrom src.train import ROOT\\nfrom src.vit_model import ViTBinaryClassifier\\n\\n\\ndef main():\\n    torch.set_num_threads(4)\\n    run = ROOT / \'outputs/models/ViT_baseline_seed42_20261115\'\\n    checkpoint_path = run / \'best.pt\'\\n    checkpoint = torch.load(checkpoint_path, weights_only=True, map_location=\'cpu\')\\n    config = checkpoint[\'config\']\\n    cutoff = json.loads((run / \'threshold_analysis/metrics.json\').read_text())[\'threshold\']\\n    processor = ViTImageProcessor.from_pretrained(run, local_files_only=True)\\n    architecture = ViTConfig.from_pretrained(\\n        config[\'pretrained_model\'], revision=config[\'revision\'],\\n        cache_dir=ROOT / \'outputs/models/hf_cache\', local_files_only=True, num_labels=1,\\n    )\\n    model = ViTBinaryClassifier(ViTForImageClassification(architecture), False)\\n    model.load_state_dict(checkpoint[\'model_state_dict\'])\\n    model.requires_grad_(False)\\n    model.eval()\\n    table = pd.read_csv(run / \'test_evaluation/predictions.csv\', dtype={\'Patient ID\': str})\\n    table[\'prediction\'] = (table.score >= cutoff).astype(int)\\n    table[\'group\'] = np.select([\\n        (table.pneumonia == 1) & (table.prediction == 1),\\n        (table.pneumonia == 0) & (table.prediction == 0),\\n        (table.pneumonia == 0) & (table.prediction == 1),\\n    ], [\'TP\', \'TN\', \'FP\'], default=\'FN\')\\n    samples = []\\n    seen = set()\\n    for group in (\'TP\', \'TN\', \'FP\', \'FN\'):\\n        candidates = table[table.group == group].sample(frac=1, random_state=42)\\n        candidates = candidates[~candidates[\'Patient ID\'].isin(seen)]\\n        chosen = candidates.drop_duplicates(\'Patient ID\').head(2)\\n        if len(chosen) != 2:\\n            raise ValueError(f\'Insufficient unique patients for {group}\')\\n        seen.update(chosen[\'Patient ID\'])\\n        samples.append(chosen)\\n    cases = pd.concat(samples).reset_index(drop=True)\\n    directory = ROOT / \'outputs/figures/gradcam\'\\n    metrics = ROOT / \'outputs/metrics/gradcam\'\\n    directory.mkdir(parents=True, exist_ok=True)\\n    metrics.mkdir(parents=True, exist_ok=True)\\n    records = []\\n    captured = {}\\n\\n    def capture(module, inputs, output):\\n        captured[\'activation\'] = output\\n\\n    # Patch tokens must influence the CLS token through the final attention block.\\n    handle = model.network.vit.encoder.layer[-1].layernorm_before.register_forward_hook(capture)\\n    try:\\n        for group in (\'TP\', \'TN\', \'FP\', \'FN\'):\\n            selected = cases[cases.group == group]\\n            fig, axes = plt.subplots(2, 3, figsize=(10, 7), layout=\'constrained\')\\n            for index, (_, row) in enumerate(selected.iterrows()):\\n                with Image.open(ROOT / \'data/raw/images\' / row[\'Image Index\']) as source:\\n                    image = source.convert(\'RGB\')\\n                    inputs = processor(image, return_tensors=\'pt\')[\'pixel_values\']\\n                    display = np.asarray(image.resize((224, 224), Image.Resampling.BILINEAR)) / 255\\n                with torch.no_grad():\\n                    reference = model(inputs)\\n                captured.clear()\\n                logits = model(inputs.requires_grad_(True))\\n                activation = captured[\'activation\']\\n                gradient = torch.autograd.grad(logits.sum(), activation)[0]\\n                torch.testing.assert_close(reference, logits)\\n                score = float(logits.detach().sigmoid().item())\\n                if abs(score - row.score) > 1e-5:\\n                    raise ValueError(\'Score differs from saved test inference\')\\n                tokens = activation[:, 1:, :]\\n                gradients = gradient[:, 1:, :]\\n                assert tokens.shape[1] == 196 and gradients.abs().sum() > 0\\n                weights = gradients.mean(dim=1, keepdim=True)\\n                raw = F.relu((weights * tokens).sum(dim=2)).reshape(1, 1, 14, 14)\\n                heat = F.interpolate(raw, (224, 224), mode=\'bilinear\', align_corners=False)[0, 0].detach().numpy()\\n                maximum = float(heat.max())\\n                if maximum > 0:\\n                    heat /= maximum\\n                assert np.isfinite(heat).all()\\n                axes[index, 0].imshow(display)\\n                axes[index, 1].imshow(heat, cmap=\'inferno\', vmin=0, vmax=1)\\n                axes[index, 2].imshow(display)\\n                overlay = axes[index, 2].imshow(heat, cmap=\'inferno\', vmin=0, vmax=1, alpha=.45)\\n                titles = [f\\"Label {row.pneumonia} | predicted {row.prediction}\\",\\n                          \'Positive-score Grad-CAM\', f\'Score {score:.4f}\']\\n                for ax, title in zip(axes[index], titles):\\n                    ax.set_title(title, fontsize=10)\\n                    ax.axis(\'off\')\\n                axes[index, 0].set_xlabel(row[\'Image Index\'])\\n                records.append({**row.to_dict(), \'recomputed_score\': score,\\n                                \'cam_max_before_normalization\': maximum})\\n                print(f\'{group}: {row[\\"Image Index\\"]}, score={score:.6f}, CAM max={maximum:.6g}\', flush=True)\\n                del activation, gradient, tokens, gradients, logits\\n                captured.clear()\\n            fig.suptitle(f\'ViT test examples | {group} | threshold {cutoff:.4f}\')\\n            fig.colorbar(overlay, ax=axes[:, 2].tolist(), shrink=.6, label=\'Within-image relative contribution\')\\n            fig.savefig(directory / f\'vit_test_{group}.png\', dpi=150)\\n            plt.close(fig)\\n    finally:\\n        handle.remove()\\n    pd.DataFrame(records).to_csv(metrics / \'vit_test_cases.csv\', index=False)\\n    (metrics / \'vit_method.json\').write_text(json.dumps({\\n        \'model\': config[\'pretrained_model\'], \'checkpoint_epoch\': checkpoint[\'epoch\'],\\n        \'checkpoint_sha256\': hashlib.sha256(checkpoint_path.read_bytes()).hexdigest(),\\n        \'threshold\': cutoff, \'cohort\': \'test\',\\n        \'selection\': \'seed 42, 2 per TP/TN/FP/FN, 8 distinct patients; qualitative examples\',\\n        \'layer\': \'vit.encoder.layer[-1].layernorm_before\',\\n        \'target\': \'positive logit for all groups\',\\n        \'method\': \'drop CLS token; spatial mean gradient per channel; weighted activations; ReLU; 14x14 to 224x224 bilinear\',\\n        \'normalization\': \'per image maximum; zero maps retained\',\\n        \'checks\': \'logits match no-grad forward; scores match saved test within 1e-5; nonzero patch gradients; finite maps\',\\n    }, indent=2), encoding=\'utf-8\')\\n\\n\\nif __name__ == \'__main__\':\\n    main()\\n", "model.py": "\\"\\"\\"DenseNet-121 baseline: frozen ImageNet backbone and one binary logit.\\"\\"\\"\\nfrom pathlib import Path\\nimport torch\\nfrom torch import nn\\nfrom torchvision.models import densenet121, DenseNet121_Weights\\n\\n\\ndef build_model(pretrained=True, freeze_backbone=True):\\n    # Keep downloaded weights inside the ignored project model directory.\\n    cache=Path(__file__).resolve().parents[1]/\'outputs/models/torch_cache\'\\n    torch.hub.set_dir(str(cache))\\n    weights=DenseNet121_Weights.IMAGENET1K_V1 if pretrained else None\\n    model=densenet121(weights=weights)\\n    if freeze_backbone:\\n        for parameter in model.parameters():\\n            parameter.requires_grad=False\\n    model.classifier=nn.Linear(model.classifier.in_features,1)\\n    return model\\n\\n\\ndef set_classifier_training(model):\\n    \\"\\"\\"Train the classifier while keeping backbone BatchNorm statistics fixed.\\"\\"\\"\\n    model.train()\\n    model.features.eval()\\n", "preprocessing.py": "\\"\\"\\"Deterministic preprocessing; CLAHE is an optional comparison condition.\\"\\"\\"\\nimport math\\nimport numpy as np\\nfrom PIL import Image\\nfrom torchvision import transforms\\nfrom torchvision.transforms import InterpolationMode\\n\\n\\ndef to_rgb(image):\\n    \\"\\"\\"Normalize Pillow image mode without modifying the source file.\\"\\"\\"\\n    return image.convert(\'RGB\')\\n\\n\\nclass ApplyCLAHE:\\n    \\"\\"\\"Apply OpenCV CLAHE to 8-bit grayscale before replicating RGB channels.\\"\\"\\"\\n    def __init__(self, clip_limit=2.0, tile_grid_size=(8, 8)):\\n        if not math.isfinite(clip_limit) or clip_limit <= 0:\\n            raise ValueError(\'clip_limit must be finite and positive.\')\\n        if len(tile_grid_size) != 2 or any(not isinstance(x, int) or x < 1 for x in tile_grid_size):\\n            raise ValueError(\'tile_grid_size must contain two positive integers.\')\\n        self.clip_limit = float(clip_limit)\\n        self.tile_grid_size = tuple(tile_grid_size)\\n\\n    def __call__(self, image):\\n        import cv2\\n        grayscale = np.asarray(image.convert(\'L\'), dtype=np.uint8)\\n        equalizer = cv2.createCLAHE(clipLimit=self.clip_limit, tileGridSize=self.tile_grid_size)\\n        adjusted = equalizer.apply(grayscale)\\n        return Image.fromarray(adjusted).convert(\'RGB\')\\n\\n\\ndef build_transform(image_size=224, *, use_clahe=False, clip_limit=2.0, tile_grid_size=(8, 8)):\\n    \\"\\"\\"No cropping or augmentation. Default retains the no-CLAHE baseline.\\n\\n    Order with CLAHE: source grayscale -> CLAHE -> RGB -> resize -> tensor -> normalize.\\n    The same CLAHE configuration must be used for training and inference if selected.\\n    \\"\\"\\"\\n    mode_transform = ApplyCLAHE(clip_limit, tile_grid_size) if use_clahe else to_rgb\\n    return transforms.Compose([\\n        transforms.Lambda(mode_transform),\\n        transforms.Resize((image_size, image_size), interpolation=InterpolationMode.BILINEAR, antialias=True),\\n        transforms.ToTensor(),\\n        transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),\\n    ])\\n\\n\\ndef build_eval_transform(image_size=224, *, use_clahe=False, clip_limit=2.0, tile_grid_size=(8, 8)):\\n    \\"\\"\\"Deterministic preprocessing for validation/test; no random augmentation.\\"\\"\\"\\n    return build_transform(image_size, use_clahe=use_clahe, clip_limit=clip_limit, tile_grid_size=tile_grid_size)\\n\\n\\ndef build_train_transform(image_size=224, *, augment=True, use_clahe=False, clip_limit=2.0, tile_grid_size=(8, 8)):\\n    \\"\\"\\"Optional weak augmentation after resizing, before tensor normalization.\\n\\n    Rotation: probability 0.5, angle uniform in [-5, 5] degrees, black fill.\\n    Brightness/contrast: probability 0.5, each factor in [0.9, 1.1].\\n    No flips, random crops, noise, erasing or color hue/saturation changes.\\n    Rotation keeps the output canvas size; small corner regions may be lost.\\n    \\"\\"\\"\\n    base = build_eval_transform(image_size, use_clahe=use_clahe, clip_limit=clip_limit, tile_grid_size=tile_grid_size)\\n    if not augment:\\n        return base\\n    augmentation = [\\n        transforms.RandomApply([\\n            transforms.RandomRotation(degrees=5, interpolation=InterpolationMode.BILINEAR, expand=False, fill=0),\\n        ], p=0.5),\\n        transforms.RandomApply([\\n            transforms.ColorJitter(brightness=0.1, contrast=0.1),\\n        ], p=0.5),\\n    ]\\n    # The deterministic pipeline has mode conversion + resize before tensor/normalize.\\n    return transforms.Compose(base.transforms[:2] + augmentation + base.transforms[2:])\\n", "threshold_analysis.py": "\\"\\"\\"Select a ViT score cutoff on validation and apply it once to test.\\"\\"\\"\\nimport argparse\\nimport json\\nfrom pathlib import Path\\n\\nimport numpy as np\\nimport pandas as pd\\nfrom sklearn.metrics import confusion_matrix, roc_curve\\n\\n\\ndef read_predictions(run, split):\\n    directory = run / f\'{split}_evaluation\'\\n    summary = json.loads((directory / \'metrics.json\').read_text(encoding=\'utf-8\'))\\n    rows = pd.read_csv(directory / \'predictions.csv\', dtype={\'Patient ID\': str})\\n    if (summary[\'split\'] != split or len(rows) != summary[\'images\']\\n            or rows[\'pneumonia\'].sum() != summary[\'positive_images\']\\n            or set(rows[\'pneumonia\']) != {0, 1}\\n            or not rows[\'score\'].between(0, 1).all()):\\n        raise ValueError(f\'Invalid {split} predictions.\')\\n    return rows, summary\\n\\n\\ndef at_threshold(rows, threshold):\\n    tn, fp, fn, tp = map(int, confusion_matrix(\\n        rows[\'pneumonia\'], rows[\'score\'] >= threshold, labels=[0, 1]\\n    ).ravel())\\n    return {\\n        \'tn\': tn, \'fp\': fp, \'fn\': fn, \'tp\': tp,\\n        \'sensitivity\': tp / (tp + fn),\\n        \'specificity\': tn / (tn + fp),\\n        \'precision\': tp / (tp + fp) if tp + fp else None,\\n        \'f1\': 2 * tp / (2 * tp + fp + fn),\\n    }\\n\\n\\ndef main():\\n    parser = argparse.ArgumentParser(description=__doc__)\\n    parser.add_argument(\'run_dir\', type=Path)\\n    args = parser.parse_args()\\n\\n    validation, val_summary = read_predictions(args.run_dir, \'val\')\\n    fpr, tpr, thresholds = roc_curve(\\n        validation[\'pneumonia\'], validation[\'score\'], drop_intermediate=False\\n    )\\n    finite = np.isfinite(thresholds)\\n    curve = pd.DataFrame({\\n        \'threshold\': thresholds[finite], \'sensitivity\': tpr[finite],\\n        \'specificity\': 1 - fpr[finite], \'youden_j\': (tpr - fpr)[finite],\\n    })\\n    # ROC thresholds descend, so take the highest threshold among numerical ties.\\n    best = curve[np.isclose(curve[\'youden_j\'], curve[\'youden_j\'].max(),\\n                            atol=1e-12, rtol=0)].iloc[0]\\n    threshold = float(best[\'threshold\'])\\n    result = {\\n        \'selection\': \'Maximum validation Youden J; highest threshold on ties\',\\n        \'checkpoint_epoch\': val_summary[\'checkpoint_epoch\'],\\n        \'threshold\': threshold,\\n        \'validation\': at_threshold(validation, threshold),\\n    }\\n\\n    test, test_summary = read_predictions(args.run_dir, \'test\')\\n    if val_summary[\'checkpoint_epoch\'] != test_summary[\'checkpoint_epoch\']:\\n        raise ValueError(\'Validation and test checkpoint epochs differ.\')\\n    if set(validation[\'Patient ID\']) & set(test[\'Patient ID\']):\\n        raise ValueError(\'Validation and test patient IDs overlap.\')\\n    result[\'test\'] = at_threshold(test, threshold)\\n    output = args.run_dir / \'threshold_analysis\'\\n    output.mkdir(exist_ok=False)\\n    curve.to_csv(output / \'validation_thresholds.csv\', index=False)\\n    (output / \'metrics.json\').write_text(json.dumps(result, indent=2), encoding=\'utf-8\')\\n    print(json.dumps(result, indent=2), flush=True)\\n\\n\\nif __name__ == \'__main__\':\\n    main()\\n", "train.py": "\\"\\"\\"CPU DenseNet and ViT training. Run from the repository with python -m src.train.\\"\\"\\"\\nimport argparse\\nimport hashlib\\nimport json\\nimport random\\nimport time\\nfrom pathlib import Path\\n\\nimport numpy as np\\nimport pandas as pd\\nimport torch\\nfrom sklearn.metrics import average_precision_score, roc_auc_score\\nfrom torch import nn\\nfrom torch.utils.data import DataLoader, Subset\\n\\nfrom src.dataset import ChestXrayDataset\\nfrom src.model import build_model, set_classifier_training\\nfrom src.preprocessing import ApplyCLAHE, build_transform\\n\\nROOT = Path(__file__).resolve().parents[1]\\n\\n\\ndef run_epoch(model, loader, criterion, optimizer=None, set_training=set_classifier_training):\\n    if optimizer is None:\\n        model.eval()\\n    else:\\n        set_training(model)\\n    loss_sum, targets_all, scores_all = 0.0, [], []\\n    with torch.set_grad_enabled(optimizer is not None):\\n        for step, (images, targets) in enumerate(loader, 1):\\n            if optimizer is not None:\\n                optimizer.zero_grad(set_to_none=True)\\n            device = next(model.parameters()).device\\n            images, targets = images.to(device), targets.to(device)\\n            logits = model(images).squeeze(1)\\n            loss = criterion(logits, targets)\\n            if logits.shape != targets.shape or not torch.isfinite(loss):\\n                raise RuntimeError(\'Invalid model output or non-finite loss\')\\n            if optimizer is not None:\\n                loss.backward()\\n                optimizer.step()\\n            loss_sum += loss.item() * len(targets)\\n            targets_all.extend(targets.detach().cpu().tolist())\\n            scores_all.extend(logits.detach().sigmoid().cpu().tolist())\\n            if step % 100 == 0:\\n                print(f\'  batch {step}/{len(loader)}\', flush=True)\\n    return {\\n        \'loss\': loss_sum / len(targets_all),\\n        \'auroc\': float(roc_auc_score(targets_all, scores_all)) if len(set(targets_all)) == 2 else None,\\n        \'auprc\': float(average_precision_score(targets_all, scores_all)) if len(set(targets_all)) == 2 else None,\\n        \'images\': len(targets_all),\\n    }\\n\\n\\ndef save_checkpoint(path, checkpoint):\\n    temporary = path.with_suffix(\'.tmp\')\\n    torch.save(checkpoint, temporary)\\n    temporary.replace(path)\\n\\n\\ndef main():\\n    parser = argparse.ArgumentParser(description=__doc__)\\n    parser.add_argument(\'--model\', choices=(\'densenet\', \'vit\'), default=\'densenet\')\\n    parser.add_argument(\'--run-name\')\\n    parser.add_argument(\'--vit-last-block\', action=\'store_true\', help=\'Fine-tune the last ViT block and final LayerNorm\')\\n    parser.add_argument(\'--epochs\', type=int, default=3)\\n    parser.add_argument(\'--batch-size\', type=int, default=8)\\n    parser.add_argument(\'--lr\', type=float)\\n    parser.add_argument(\'--threads\', type=int, default=4)\\n    parser.add_argument(\'--seed\', type=int, default=42)\\n    parser.add_argument(\'--smoke\', action=\'store_true\', help=\'Use only 32 training images for execution checks\')\\n    parser.add_argument(\'--output-root\', help=\'Checkpoint parent directory; defaults to outputs/models\')\\n    parser.add_argument(\'--clahe\', action=\'store_true\')\\n    parser.add_argument(\'--device\', choices=(\'cpu\', \'cuda\'), default=\'cpu\')\\n    args = parser.parse_args()\\n    if args.device == \'cuda\' and not torch.cuda.is_available():\\n        parser.error(\'CUDA GPU is unavailable\')\\n    device = torch.device(args.device)\\n    if args.device == \'cuda\':\\n        torch.backends.cuda.matmul.allow_tf32 = False\\n        torch.backends.cudnn.allow_tf32 = False\\n    is_vit = args.model == \'vit\'\\n    if args.vit_last_block and not is_vit:\\n        parser.error(\'--vit-last-block requires --model vit\')\\n    if args.lr is None:\\n        args.lr = 1e-4 if args.vit_last_block else 1e-3\\n    if args.run_name is None:\\n        args.run_name = \'ViT_baseline_seed42\' if is_vit else \'B0_baseline_seed42\'\\n    if min(args.epochs, args.batch_size, args.threads) < 1 or not np.isfinite(args.lr) or args.lr <= 0:\\n        parser.error(\'Epochs, batch size, threads and learning rate must be positive\')\\n    if Path(args.run_name).name != args.run_name or args.run_name in (\'.\', \'..\'):\\n        parser.error(\'Run name must be a single directory name\')\\n    random.seed(args.seed)\\n    np.random.seed(args.seed)\\n    torch.manual_seed(args.seed)\\n    torch.set_num_threads(args.threads)\\n    torch.use_deterministic_algorithms(True)\\n    manifest_dir = ROOT / \'data/processed/reduced_train_full_eval_qc\'\\n    set_training = set_classifier_training\\n    model_config = {}\\n    if is_vit:\\n        from src.vit_model import MODEL_ID, MODEL_REVISION, load_vit\\n\\n        model, processor = load_vit(ROOT / \'outputs/models/hf_cache\', args.vit_last_block)\\n\\n        clahe = ApplyCLAHE()\\n        def transform(image):\\n            if args.clahe:\\n                image = clahe(image)\\n            return processor(image.convert(\'RGB\'), return_tensors=\'pt\')[\'pixel_values\'][0]\\n\\n        set_training = type(model).train_selected_layers\\n        model_config = dict(pretrained_model=MODEL_ID, revision=MODEL_REVISION)\\n\\n    else:\\n        transform = build_transform(use_clahe=args.clahe)\\n\\n    train = ChestXrayDataset(manifest_dir / \'train.csv\', ROOT / \'data/raw/images\', transform)\\n    hashes = {\'train\': hashlib.sha256((manifest_dir / \'train.csv\').read_bytes()).hexdigest()}\\n    if args.smoke:\\n        # Execution checks never open validation or test images.\\n        indices = torch.randperm(len(train), generator=torch.Generator().manual_seed(args.seed))[:32].tolist()\\n        train = Subset(train, indices)\\n        val = train\\n    else:\\n        val = ChestXrayDataset(manifest_dir / \'val.csv\', ROOT / \'data/raw/images\', transform)\\n        if set(train.table[\'Patient ID\']) & set(val.table[\'Patient ID\']):\\n            raise ValueError(\'Training and validation patients overlap\')\\n        hashes[\'val\'] = hashlib.sha256((manifest_dir / \'val.csv\').read_bytes()).hexdigest()\\n    output_root = Path(args.output_root) if args.output_root else ROOT / \'outputs/models\'\\n    output = output_root / args.run_name\\n    output.mkdir(parents=True, exist_ok=False)\\n    config = dict(vars(args), manifest_sha256=hashes, augmentation=False, clahe=args.clahe,\\n                  clahe_parameters={\'clip_limit\': 2.0, \'tile_grid_size\': [8, 8]} if args.clahe else None,\\n                  selection=\'validation AUROC, earliest epoch on ties\', device=args.device,\\n                  torch_version=str(torch.__version__), train_images=len(train), val_images=len(val),\\n                  smoke_metrics_are_not_validation=args.smoke)\\n    config.update(model_config)\\n    if is_vit:\\n        processor.save_pretrained(output)\\n    (output / \'config.json\').write_text(json.dumps(config, indent=2), encoding=\'utf-8\')\\n    generator = torch.Generator().manual_seed(args.seed)\\n    train_loader = DataLoader(train, batch_size=args.batch_size, shuffle=True, generator=generator, num_workers=0)\\n    val_loader = DataLoader(val, batch_size=args.batch_size, shuffle=False, num_workers=0)\\n    if not is_vit:\\n        model = build_model()\\n    model.to(device)\\n    optimizer = torch.optim.Adam((p for p in model.parameters() if p.requires_grad), lr=args.lr)\\n    criterion = nn.BCEWithLogitsLoss()\\n    if args.smoke:\\n        trainable_names = {name for name, p in model.named_parameters() if p.requires_grad}\\n        frozen = {name: value.clone() for name, value in model.state_dict().items()\\n                  if name not in trainable_names}\\n        if args.vit_last_block:\\n            block_parameter = model.network.vit.encoder.layer[-1].output.dense.weight\\n            block_before = block_parameter.detach().clone()\\n    classifier_before = model.classifier.weight.detach().clone()\\n    history, best = [], -float(\'inf\')\\n    for epoch in range(1, args.epochs + 1):\\n        started = time.perf_counter()\\n        print(f\'Epoch {epoch}/{args.epochs}: training {len(train)} images\', flush=True)\\n        training = run_epoch(model, train_loader, criterion, optimizer, set_training)\\n        print(f\'Epoch {epoch}: evaluating {len(val)} images\', flush=True)\\n        validation = run_epoch(model, val_loader, criterion)\\n        row = {\'epoch\': epoch, \'seconds\': time.perf_counter() - started,\\n               **{f\'train_{k}\': v for k, v in training.items()},\\n               **{f\'val_{k}\': v for k, v in validation.items()}}\\n        history.append(row)\\n        pd.DataFrame(history).to_csv(output / \'history.csv\', index=False)\\n        checkpoint = {\'model_state_dict\': model.state_dict(), \'optimizer_state_dict\': optimizer.state_dict(),\\n                      \'epoch\': epoch, \'config\': config, \'metrics\': row,\\n                      \'torch_rng_state\': torch.get_rng_state(), \'loader_rng_state\': generator.get_state()}\\n        save_checkpoint(output / \'last.pt\', checkpoint)\\n        if validation[\'auroc\'] is not None and validation[\'auroc\'] > best:\\n            best = validation[\'auroc\']\\n            save_checkpoint(output / \'best.pt\', checkpoint)\\n        print(json.dumps(row), flush=True)\\n    if args.smoke:\\n        state = model.state_dict()\\n        assert all(torch.equal(value, state[key]) for key, value in frozen.items())\\n        if args.vit_last_block:\\n            assert not torch.equal(block_before, block_parameter)\\n        del frozen, state\\n        assert not torch.equal(classifier_before, model.classifier.weight)\\n        if is_vit:\\n            from src.vit_model import ViTBinaryClassifier, ViTForImageClassification\\n\\n            restored = ViTBinaryClassifier(\\n                ViTForImageClassification(model.network.config), args.vit_last_block\\n            )\\n        else:\\n            restored = build_model(pretrained=False)\\n        restored.load_state_dict(torch.load(output / \'last.pt\', weights_only=True, map_location=\'cpu\')[\'model_state_dict\'])\\n        restored.to(device)\\n        restored.eval()\\n        images, _ = next(iter(val_loader))\\n        with torch.inference_mode():\\n            images = images.to(device)\\n            torch.testing.assert_close(model(images), restored(images), rtol=0, atol=0)\\n        print(\'SMOKE PASSED: frozen weights/buffers, trainable weights updated, exact checkpoint reload.\', flush=True)\\n    (output / \'completed.json\').write_text(json.dumps({\'epochs\': args.epochs, \'best_auroc\': best if np.isfinite(best) else None}), encoding=\'utf-8\')\\n    print(f\'Completed: {output}\', flush=True)\\n\\n\\nif __name__ == \'__main__\':\\n    main()\\n", "vit_model.py": "\\"\\"\\"Hugging Face ViT for one-logit pneumonia label classification.\\"\\"\\"\\nfrom torch import nn\\nfrom transformers import ViTForImageClassification, ViTImageProcessor\\n\\nMODEL_ID = \\"google/vit-base-patch16-224\\"\\nMODEL_REVISION = \\"3f49326eb077187dfe1c2a2bb15fbd74e6ab91e3\\"\\n\\n\\nclass ViTBinaryClassifier(nn.Module):\\n    def __init__(self, network, unfreeze_last_block):\\n        super().__init__()\\n        self.network = network\\n        self.unfreeze_last_block = unfreeze_last_block\\n        network.vit.requires_grad_(False)\\n        if unfreeze_last_block:\\n            network.vit.encoder.layer[-1].requires_grad_(True)\\n            network.vit.layernorm.requires_grad_(True)\\n\\n    @property\\n    def classifier(self):\\n        return self.network.classifier\\n\\n    def forward(self, images):\\n        return self.network(pixel_values=images).logits\\n\\n    def train_selected_layers(self):\\n        self.eval()\\n        self.classifier.train()\\n        if self.unfreeze_last_block:\\n            self.network.vit.encoder.layer[-1].train()\\n            self.network.vit.layernorm.train()\\n\\n\\ndef load_vit(cache_dir, unfreeze_last_block):\\n    processor = ViTImageProcessor.from_pretrained(MODEL_ID, cache_dir=cache_dir, revision=MODEL_REVISION)\\n    network = ViTForImageClassification.from_pretrained(\\n        MODEL_ID, cache_dir=cache_dir, revision=MODEL_REVISION,\\n        num_labels=1, ignore_mismatched_sizes=True, use_safetensors=True,\\n    )\\n    return ViTBinaryClassifier(network, unfreeze_last_block), processor\\n", "__init__.py": ""}')
for name, content in sources.items():
    (workspace / 'src' / name).write_text(content, encoding='utf-8')


In [ ]:
import hashlib
import pandas as pd
import json
import subprocess
import sys
import time
baseline_config = json.loads((workspace / 'reference_vit/config.json').read_text())
manifest_root = workspace / 'data/processed/reduced_train_full_eval_qc'
for split in ('train', 'val'):
    manifest = manifest_root / f'{split}.csv'
    assert hashlib.sha256(manifest.read_bytes()).hexdigest() == baseline_config['manifest_sha256'][split]
    rows = pd.read_csv(manifest)
    missing = [name for name in rows['Image Index'] if not (workspace / 'data/raw/images' / name).is_file()]
    assert not missing, missing[:5]
subprocess.run([sys.executable, '-m', 'pip', 'freeze'], stdout=(output_root / 'environment.txt').open('w'), check=True)
print('학습 4,076장·검증 16,872장 파일과 목록 해시 확인 완료')


In [ ]:
# Extend both matched ViT conditions to 10 total epochs.
import json, subprocess, sys
from pathlib import Path
(workspace / 'src/extend_training.py').write_text("\"\"\"Extend a ViT run from its last completed epoch into a separate output directory.\"\"\"\nimport argparse\nimport hashlib\nimport json\nimport random\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch import nn\nfrom torch.utils.data import DataLoader, Subset\n\nfrom src.dataset import ChestXrayDataset\nfrom src.preprocessing import ApplyCLAHE\nfrom src.train import ROOT, run_epoch, save_checkpoint\nfrom src.vit_model import load_vit\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument('source', type=Path)\n    parser.add_argument('--epochs', type=int, default=10)\n    parser.add_argument('--output', type=Path, required=True)\n    args = parser.parse_args()\n    source = args.source.resolve()\n    output = args.output.resolve()\n    if source == output:\n        parser.error('Use a distinct output directory.')\n    # Retry an interrupted extension from its own last completed epoch.\n    origin = output if (output / 'last.pt').is_file() else source\n    checkpoint = torch.load(origin / 'last.pt', weights_only=True, map_location='cpu')\n    config = checkpoint['config'].copy()\n    if config.get('model') != 'vit':\n        parser.error('Only ViT extensions are supported.')\n    if args.epochs <= checkpoint['epoch']:\n        parser.error('Target epochs must exceed the saved epoch.')\n    device = torch.device(config['device'])\n    if device.type == 'cuda' and not torch.cuda.is_available():\n        parser.error('Source run requires a CUDA GPU.')\n    random.seed(config['seed'])\n    np.random.seed(config['seed'])\n    torch.manual_seed(config['seed'])\n    torch.set_num_threads(config['threads'])\n    torch.use_deterministic_algorithms(True)\n    if device.type == 'cuda':\n        torch.backends.cuda.matmul.allow_tf32 = False\n        torch.backends.cudnn.allow_tf32 = False\n\n    manifest_dir = ROOT / 'data/processed/reduced_train_full_eval_qc'\n    for split, digest in config['manifest_sha256'].items():\n        if hashlib.sha256((manifest_dir / f'{split}.csv').read_bytes()).hexdigest() != digest:\n            raise ValueError('Manifest changed.')\n    model, processor = load_vit(ROOT / 'outputs/models/hf_cache', config['vit_last_block'])\n    clahe = ApplyCLAHE()\n\n    def transform(image):\n        if config['clahe']:\n            image = clahe(image)\n        return processor(image.convert('RGB'), return_tensors='pt')['pixel_values'][0]\n\n    train = ChestXrayDataset(manifest_dir / 'train.csv', ROOT / 'data/raw/images', transform)\n    if config['smoke']:\n        indices = torch.randperm(len(train), generator=torch.Generator().manual_seed(config['seed']))[:32].tolist()\n        train = Subset(train, indices)\n        val = train\n    else:\n        val = ChestXrayDataset(manifest_dir / 'val.csv', ROOT / 'data/raw/images', transform)\n        assert not set(train.table['Patient ID']) & set(val.table['Patient ID'])\n    assert len(train) == config['train_images'] and len(val) == config['val_images']\n    generator = torch.Generator()\n    generator.set_state(checkpoint['loader_rng_state'])\n    train_loader = DataLoader(train, batch_size=config['batch_size'], shuffle=True, generator=generator, num_workers=0)\n    val_loader = DataLoader(val, batch_size=config['batch_size'], shuffle=False, num_workers=0)\n    model.load_state_dict(checkpoint['model_state_dict'])\n    model.to(device)\n    optimizer = torch.optim.Adam((p for p in model.parameters() if p.requires_grad), lr=config['lr'])\n    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])\n    torch.set_rng_state(checkpoint['torch_rng_state'])\n    if device.type == 'cuda' and 'cuda_rng_states' in checkpoint:\n        torch.cuda.set_rng_state_all(checkpoint['cuda_rng_states'])\n    history = pd.read_csv(origin / 'history.csv')\n    history = history[history.epoch <= checkpoint['epoch']].to_dict('records')\n    if [int(row['epoch']) for row in history] != list(range(1, checkpoint['epoch'] + 1)):\n        raise ValueError('Incomplete source history.')\n    best_checkpoint = torch.load(origin / 'best.pt', weights_only=True, map_location='cpu')\n    best = best_checkpoint['metrics']['val_auroc']\n    start_epoch = checkpoint['epoch'] + 1\n    if output.exists() and origin != output:\n        parser.error('Output already exists without a resumable checkpoint.')\n    output.mkdir(parents=True, exist_ok=True)\n    config.update(epochs=args.epochs, run_name=output.name, resumed_from=str(source),\n                  continuation_start_epoch=start_epoch)\n    processor.save_pretrained(output)\n    (output / 'config.json').write_text(json.dumps(config, indent=2), encoding='utf-8')\n    if origin != output:\n        save_checkpoint(output / 'best.pt', best_checkpoint)\n    del checkpoint, best_checkpoint\n    criterion = nn.BCEWithLogitsLoss()\n    for epoch in range(start_epoch, args.epochs + 1):\n        started = time.perf_counter()\n        print(f'Epoch {epoch}/{args.epochs}, CLAHE={config[\"clahe\"]}', flush=True)\n        training = run_epoch(model, train_loader, criterion, optimizer, type(model).train_selected_layers)\n        validation = run_epoch(model, val_loader, criterion)\n        row = dict(epoch=epoch, seconds=time.perf_counter() - started,\n                   **{f'train_{key}': value for key, value in training.items()},\n                   **{f'val_{key}': value for key, value in validation.items()})\n        history.append(row)\n        pd.DataFrame(history).to_csv(output / 'history.csv', index=False)\n        checkpoint = dict(model_state_dict=model.state_dict(), optimizer_state_dict=optimizer.state_dict(),\n                          epoch=epoch, config=config, metrics=row,\n                          torch_rng_state=torch.get_rng_state(), loader_rng_state=generator.get_state())\n        if device.type == 'cuda':\n            checkpoint['cuda_rng_states'] = torch.cuda.get_rng_state_all()\n        if row['val_auroc'] is not None and row['val_auroc'] > best:\n            best = row['val_auroc']\n            save_checkpoint(output / 'best.pt', checkpoint)\n        save_checkpoint(output / 'last.pt', checkpoint)\n        print(json.dumps(row), flush=True)\n    (output / 'completed.json').write_text(json.dumps(dict(epochs=args.epochs, best_auroc=best)), encoding='utf-8')\n    print(f'Completed {output}', flush=True)\n\n\nif __name__ == '__main__':\n    main()\r\n", encoding='utf-8')

def run_logged(command, path):
    print('RUN', ' '.join(map(str, command)), flush=True)
    with open(path, 'a', encoding='utf-8') as log:
        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end='', flush=True)
            log.write(line)
            log.flush()
        if process.wait() != 0:
            raise RuntimeError(f'Command failed; see {path}')

extended_runs = []
for base_name, clahe in [('ViT_GPU_B0_seed42', False), ('ViT_GPU_CLAHE_seed42', True)]:
    source = output_root / base_name
    assert (source / 'last.pt').exists() and (source / 'history.csv').exists(), f'Missing checkpoint: {source}'
    target = output_root / (base_name + '_10epochs')
    if not (target / 'completed.json').exists():
        run_logged([sys.executable, '-u', '-m', 'src.extend_training', str(source), '--epochs', '10', '--output', str(target)], output_root / (base_name + '_extension.log'))
    assert json.loads((target / 'completed.json').read_text())['epochs'] == 10
    extended_runs.append((target, clahe))

summary = []
for run, clahe in extended_runs:
    if not (run / 'val_evaluation/metrics.json').exists():
        run_logged([sys.executable, '-u', '-m', 'src.evaluate', str(run), '--split', 'val', '--device', 'cuda'], output_root / (run.name + '_evaluation.log'))
    metrics = json.loads((run / 'val_evaluation/metrics.json').read_text())
    h = pd.read_csv(run / 'history.csv')
    summary.append(dict(run=run.name, clahe=clahe, best_epoch=metrics['checkpoint_epoch'], val_auroc=metrics['auroc'], val_average_precision=metrics['average_precision'], training_and_validation_minutes=h.seconds.sum()/60))
comparison_10 = pd.DataFrame(summary)
comparison_10.to_csv(output_root / 'clahe_comparison_10epochs.csv', index=False)
display(comparison_10)
print('10-epoch comparison completed', flush=True)
